[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_05_classical_ml/ch25_unsupervised_clustering_pca/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 25 Coding Exam — Unsupervised Clustering & PCA Audit (100 Points)

**Book Reference:** Chapter 25, Section 25.7 (Chapter Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Unsupervised ML Pipeline | Points |
| :--- | :--- | :---: |
| **Task 25.1** | `select_optimal_k_silhouette(X, k_range, seed=42)` (Standardize + K-sweep + peak silhouette) | 30 |
| **Task 25.2** | `district_risk_phenotyping(clinic_df, n_clusters=3, seed=42)` (District aggregation + scaling + K-Means) | 35 |
| **Task 25.3** | `pca_clinical_compression_audit(X, variance_threshold=0.85)` (Scale + PCA + PC1 loading + reconstruction MSE) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 25 Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import sharukh_tutor as tutor

tutor.activate(chapter="ch25", notebook="coding_exam")


---
## Exam Task 25.1 (30 Points): `select_optimal_k_silhouette(X, k_range, seed=42)`

Without mutating `X`:
1. Standardize `X` via `StandardScaler().fit_transform(X)`.
2. Sweep each `k` in `k_range` using `KMeans(n_clusters=k, n_init=10, random_state=seed)`.
3. Build `table` as a `pd.DataFrame` with columns `["k", "inertia", "silhouette"]` rounded to **4 decimals**.
4. Return `{"best_k": int, "best_silhouette": round4, "table": table}` selecting the `k` with maximum silhouette score.

In [ ]:
def select_optimal_k_silhouette(X, k_range, seed=42):
    """Standardize X, sweep k_range, and pick k maximizing silhouette."""
    X_sc = StandardScaler().fit_transform(X)
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = km.fit_predict(X_sc)
        sil = silhouette_score(X_sc, labels)
        rows.append({
            "k": int(k),
            "inertia": round(float(km.inertia_), 4),
            "silhouette": round(float(sil), 4),
        })
    table = pd.DataFrame(rows)
    best = table.loc[table["silhouette"].idxmax()]
    return {
        "best_k": int(best["k"]),
        "best_silhouette": round(float(best["silhouette"]), 4),
        "table": table,
    }


---
## Exam Task 25.2 (35 Points): `district_risk_phenotyping(clinic_df, n_clusters=3, seed=42)`

Cluster districts in `clinic_df` (without mutating `clinic_df`) into clinical risk phenotypes:
1. Group `clinic_df` by `"district"` (`as_index=False`) and compute four summary indicators:
   - `mean_temp`: mean of `temp_c`
   - `mean_sbp`: mean of `systolic_bp`
   - `mean_hb`: mean of `hemoglobin_g_dl`
   - `urgent_rate`: proportion of non-routine triage cases (`(triage_priority != "Routine").mean()`)
2. Standardize those four indicators with `StandardScaler()`.
3. Fit `KMeans(n_clusters=n_clusters, n_init=10, random_state=seed)` and store integer labels in a `"cluster"` column.
4. Return `{"district_clusters": df, "silhouette": round4}`.

In [ ]:
def district_risk_phenotyping(clinic_df, n_clusters=3, seed=42):
    """Aggregate district triage profiles, scale, and cluster."""
    agg = clinic_df.groupby("district", as_index=False).agg(
        mean_temp=("temp_c", "mean"),
        mean_sbp=("systolic_bp", "mean"),
        mean_hb=("hemoglobin_g_dl", "mean"),
        urgent_rate=(
            "triage_priority", lambda s: (s != "Routine").mean()
        ),
    )
    cols = ["mean_temp", "mean_sbp", "mean_hb", "urgent_rate"]
    X_sc = StandardScaler().fit_transform(agg[cols].to_numpy(dtype=float))
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed)
    labels = km.fit_predict(X_sc).astype(int)
    agg["cluster"] = labels
    sil = round(float(silhouette_score(X_sc, labels)), 4)
    return {
        "district_clusters": agg,
        "silhouette": sil,
    }


---
## Exam Task 25.3 (35 Points): `pca_clinical_compression_audit(X, variance_threshold=0.85)`

Audit standardized PCA compression on `X` (without mutating `X`):
1. Standardize `X` via `StandardScaler().fit_transform(X)`.
2. Fit a full `PCA()` to find the smallest integer `k` where `np.cumsum(explained_variance_ratio_)[k - 1] >= variance_threshold`.
3. Fit `PCA(n_components=k)` on `X_scaled`, project `Z = pca_k.fit_transform(X_scaled)`, and reconstruct `X_hat = pca_k.inverse_transform(Z)`.
4. Return a dictionary with exact keys:
   - `"n_components"`: `int(k)`
   - `"cumulative_variance"`: sum of retained variance ratios rounded to **4 decimals**
   - `"top_pc1_feature_idx"`: `int` index of feature with largest `abs` loading on PC1
   - `"reconstruction_mse"`: `np.mean((X_scaled - X_hat) ** 2)` rounded to **5 decimals**

In [ ]:
def pca_clinical_compression_audit(X, variance_threshold=0.85):
    """Scale X, pick PCs for variance_threshold, and audit recon MSE."""
    X_sc = StandardScaler().fit_transform(X)
    full_pca = PCA().fit(X_sc)
    cum = np.cumsum(full_pca.explained_variance_ratio_)
    k = int(np.argmax(cum >= variance_threshold) + 1)
    pca_k = PCA(n_components=k)
    Z = pca_k.fit_transform(X_sc)
    X_hat = pca_k.inverse_transform(Z)
    cum_var = round(float(np.sum(pca_k.explained_variance_ratio_)), 4)
    top_idx = int(np.argmax(np.abs(pca_k.components_[0])))
    mse = round(float(np.mean((X_sc - X_hat) ** 2)), 5)
    return {
        "n_components": k,
        "cumulative_variance": cum_var,
        "top_pc1_feature_idx": top_idx,
        "reconstruction_mse": mse,
    }


In [ ]:
exam_submissions = {
    "ch25_exam_01": select_optimal_k_silhouette,
    "ch25_exam_02": district_risk_phenotyping,
    "ch25_exam_03": pca_clinical_compression_audit,
}

tutor.check_exam("ch25", exam_submissions)


---
## Chapter 25 Coding Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 25.1 (Skipping `StandardScaler`):** Running K-Means directly on unscaled inputs allows a single high-variance column (like `age_years` or a 10,000x-scaled biomarker) to dictate every Euclidean distance, changing both the silhouette table and `best_k`.
- **Task 25.2 (District Aggregation & Elevated Triage Rate):** Grouping by `district` collapses 120 patient records into 6 sub-national health-system profiles. Computing `urgent_rate = (s != "Routine").mean()` captures elevated triage priority cleanly across datasets containing `"Moderate"` and/or `"Urgent"` flags, and scaling prevents `mean_sbp` (~125 mmHg) from dwarfing `mean_temp` (~37.4 C) and `urgent_rate` (0.37–0.57).
- **Task 25.3 (Unsigned Loadings & Unscaled Reconstruction MSE):** Because PCA eigenvectors have arbitrary sign, taking `np.argmax(pca_k.components_[0])` without `np.abs(...)` misses strong negative loadings! Similarly, comparing `X_hat = pca_k.inverse_transform(Z)` against standardized `X_sc` yields `MSE = 0.13789 = 1 - 0.86211`.